# 作业 3

姓名：陈浩  
学号：20234080133


## 2 卷积和池化层

### 2.1 理论计算题

输入图像大小为 $3 \times 32 \times 32$，卷积层包含 16 个卷积核，每个卷积核大小为 $3 \times 5 \times 5$，填充 $p=2$，步幅 $s=2$。

#### 1. 输出特征图尺寸

卷积输出的高和宽为

$$
H_{out}=W_{out}=\left\lfloor \frac{n+2p-k}{s}\right\rfloor + 1
=\left\lfloor \frac{32+2\times 2-5}{2}\right\rfloor + 1
=\left\lfloor 15.5\right\rfloor + 1=16
$$

输出通道数等于卷积核个数，因此输出特征图尺寸为

$$
\boxed{16 \times 16 \times 16}
$$

即“通道数 $\times$ 高 $\times$ 宽”为 $16 \times 16 \times 16$。

#### 2. 单个输出像素的乘法次数

单个输出通道的一个像素值由一个 $3 \times 5 \times 5$ 卷积核与输入局部窗口逐元素相乘后求和得到，因此乘法次数为

$$
3\times 5\times 5=75
$$

所以单个输出通道的一个像素需要

$$
\boxed{75}
$$

次乘法操作。若计算全部 16 个输出通道在同一空间位置上的像素，则需要 $16\times75=1200$ 次乘法。


### 2.2 编程题：手动实现二维最大池化


In [1]:
import numpy as np

try:
    import torch
    from torch import nn
    from torchvision import transforms
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc
    print("PyTorch 当前环境不可用，后续代码将使用 NumPy 演示可验证部分。")
    print("导入错误：", repr(exc))


def _pair(value):
    if isinstance(value, tuple):
        return value
    return (value, value)


def max_pool2d_manual(x, kernel_size, stride=None, padding=0):
    """只使用基础张量操作实现支持 stride 和 padding 的 2D Max Pooling。"""
    if not TORCH_AVAILABLE:
        return max_pool2d_manual_numpy(x, kernel_size, stride, padding)

    original_dim = x.dim()
    kernel_h, kernel_w = _pair(kernel_size)
    if stride is None:
        stride = kernel_size
    stride_h, stride_w = _pair(stride)
    pad_h, pad_w = _pair(padding)

    if original_dim == 2:
        x = x.unsqueeze(0).unsqueeze(0)
    elif original_dim == 3:
        x = x.unsqueeze(0)
    elif original_dim != 4:
        raise ValueError("x must be a 2D, 3D or 4D tensor")

    n, c, h, w = x.shape
    padded = x.new_full((n, c, h + 2 * pad_h, w + 2 * pad_w), float("-inf"))
    padded[:, :, pad_h:pad_h + h, pad_w:pad_w + w] = x

    out_h = (padded.shape[2] - kernel_h) // stride_h + 1
    out_w = (padded.shape[3] - kernel_w) // stride_w + 1
    y = x.new_empty((n, c, out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            window = padded[:, :, i * stride_h:i * stride_h + kernel_h, j * stride_w:j * stride_w + kernel_w]
            y[:, :, i, j] = window.amax(dim=(-2, -1))

    if original_dim == 2:
        return y.squeeze(0).squeeze(0)
    if original_dim == 3:
        return y.squeeze(0)
    return y


def max_pool2d_manual_numpy(x, kernel_size, stride=None, padding=0):
    x = np.asarray(x)
    original_dim = x.ndim
    kernel_h, kernel_w = _pair(kernel_size)
    if stride is None:
        stride = kernel_size
    stride_h, stride_w = _pair(stride)
    pad_h, pad_w = _pair(padding)

    if original_dim == 2:
        x = x[None, None, :, :]
    elif original_dim == 3:
        x = x[None, :, :, :]
    elif original_dim != 4:
        raise ValueError("x must be a 2D, 3D or 4D array")

    n, c, h, w = x.shape
    padded = np.full((n, c, h + 2 * pad_h, w + 2 * pad_w), -np.inf, dtype=float)
    padded[:, :, pad_h:pad_h + h, pad_w:pad_w + w] = x

    out_h = (padded.shape[2] - kernel_h) // stride_h + 1
    out_w = (padded.shape[3] - kernel_w) // stride_w + 1
    y = np.empty((n, c, out_h, out_w), dtype=float)

    for i in range(out_h):
        for j in range(out_w):
            window = padded[:, :, i * stride_h:i * stride_h + kernel_h, j * stride_w:j * stride_w + kernel_w]
            y[:, :, i, j] = window.max(axis=(-2, -1))

    if original_dim == 2:
        return y.squeeze(0).squeeze(0)
    if original_dim == 3:
        return y.squeeze(0)
    return y


if TORCH_AVAILABLE:
    x = torch.arange(1, 17, dtype=torch.float32).reshape(1, 1, 4, 4)
    y_manual = max_pool2d_manual(x, kernel_size=2, stride=2, padding=0)
    y_torch = nn.MaxPool2d(kernel_size=2, stride=2)(x)
    print("input:\n", x.squeeze())
    print("manual output:\n", y_manual.squeeze())
    print("same as torch:", torch.equal(y_manual, y_torch))
else:
    x = np.arange(1, 17, dtype=float).reshape(1, 1, 4, 4)
    y_manual = max_pool2d_manual(x, kernel_size=2, stride=2, padding=0)
    print("input:\n", x.squeeze())
    print("manual output:\n", y_manual.squeeze())


input:
 tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.],
        [13., 14., 15., 16.]])
manual output:
 tensor([[ 6.,  8.],
        [14., 16.]])
same as torch: True


## 3 LeNet, AlexNet, VGG 和 NiN

### 3.1 理论计算题

在 VGG 网络中，假设输入和输出的特征图通道数均为 $C$，且卷积层都不带偏置。

#### 1. 一个 $5\times5$ 卷积层的参数量

每个输出通道需要一个大小为 $C\times5\times5$ 的卷积核，共有 $C$ 个输出通道，因此参数量为

$$
C\times C\times5\times5=\boxed{25C^2}
$$

#### 2. 两个串联 $3\times3$ 卷积层的总参数量

每个 $3\times3$ 卷积层的参数量为

$$
C\times C\times3\times3=9C^2
$$

两个串联的 $3\times3$ 卷积层总参数量为

$$
2\times9C^2=\boxed{18C^2}
$$

因此，在通道数相同且不带偏置时，两个 $3\times3$ 卷积层比一个 $5\times5$ 卷积层少 $25C^2-18C^2=7C^2$ 个参数，同时还能引入两次非线性激活。


### 3.2 编程题：定义 NiN 块


In [2]:
def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    if not TORCH_AVAILABLE:
        return [
            ("Conv2d", in_channels, out_channels, kernel_size, stride, padding),
            ("ReLU",),
            ("Conv2d", out_channels, out_channels, 1, 1, 0),
            ("ReLU",),
            ("Conv2d", out_channels, out_channels, 1, 1, 0),
            ("ReLU",),
        ]
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
    )


block = nin_block(3, 16, kernel_size=5, stride=2, padding=2)
print(block)
if TORCH_AVAILABLE:
    x = torch.randn(4, 3, 32, 32)
    y = block(x)
    print("output shape:", tuple(y.shape))
else:
    out_h = (32 + 2 * 2 - 5) // 2 + 1
    print("PyTorch 可用时，该 NiN 块输出形状为:", (4, 16, out_h, out_h))


Sequential(
  (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
  (1): ReLU()
  (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)
output shape: (4, 16, 16, 16)


## 4 Inception, 批量归一化和残差网络

### 4.1 理论计算题

一个 mini-batch 中某通道某空间位置的 4 个特征值为

$$
x_1=2,\quad x_2=4,\quad x_3=6,\quad x_4=8
$$

批量归一化参数为 $\gamma=2$，$\beta=1$，$\epsilon=0$。

均值为

$$
\mu=\frac{2+4+6+8}{4}=5
$$

方差为

$$
\sigma^2=\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}
=\frac{9+1+1+9}{4}=5
$$

标准化结果为

$$
\hat{x}_i=\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}=\frac{x_i-5}{\sqrt{5}}
$$

再进行缩放和平移：

$$
y_i=\gamma \hat{x}_i+\beta=2\cdot \frac{x_i-5}{\sqrt{5}}+1
$$

所以

$$
y_1=1-\frac{6}{\sqrt5}\approx -1.6833
$$

$$
y_2=1-\frac{2}{\sqrt5}\approx 0.1056
$$

$$
y_3=1+\frac{2}{\sqrt5}\approx 1.8944
$$

$$
y_4=1+\frac{6}{\sqrt5}\approx 3.6833
$$

最终输出为

$$
\boxed{[-1.6833,\ 0.1056,\ 1.8944,\ 3.6833]}
$$


### 4.2 编程题：定义残差块 Residual


In [3]:
if TORCH_AVAILABLE:
    class Residual(nn.Module):
        def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
            super().__init__()
            self.conv1 = nn.Conv2d(input_channels, num_channels, kernel_size=3,
                                   padding=1, stride=strides)
            self.bn1 = nn.BatchNorm2d(num_channels)
            self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3,
                                   padding=1)
            self.bn2 = nn.BatchNorm2d(num_channels)
            if use_1x1conv:
                self.conv3 = nn.Conv2d(input_channels, num_channels, kernel_size=1,
                                       stride=strides)
            else:
                self.conv3 = None
            self.relu = nn.ReLU(inplace=True)

        def forward(self, x):
            y = self.relu(self.bn1(self.conv1(x)))
            y = self.bn2(self.conv2(y))
            if self.conv3 is not None:
                x = self.conv3(x)
            y += x
            return self.relu(y)


    x = torch.randn(4, 3, 32, 32)
    blk_same = Residual(3, 3)
    blk_down = Residual(3, 6, use_1x1conv=True, strides=2)
    print("same shape:", tuple(blk_same(x).shape))
    print("downsample shape:", tuple(blk_down(x).shape))
else:
    print("PyTorch 不可用；Residual 类的 PyTorch 实现已保留在本单元的 TORCH_AVAILABLE 分支中。")
    print("use_1x1conv=True, strides=2 时，示例输入 (4, 3, 32, 32) 的输出形状为 (4, 6, 16, 16)。")


same shape: (4, 3, 32, 32)
downsample shape: (4, 6, 16, 16)


## 5 图像增广，微调和样式迁移

### 5.1 理论计算题

#### 1. 为什么底层特征提取层使用较小学习率或冻结，而顶层输出层使用较大学习率？

预训练网络的底层通常学习到边缘、纹理、颜色、简单形状等通用视觉特征。这些特征在 ImageNet 和新的目标数据集之间往往可以迁移，若使用过大的学习率，容易破坏已经学到的通用表示，因此通常给底层设置较小学习率，甚至直接冻结参数。

顶层输出层更接近具体任务，原模型的分类类别与新数据集通常不同，因此顶层往往需要重新初始化并适配新类别。由于它从随机初始化开始，距离目标任务的最优参数更远，所以通常设置较大的学习率，使其更快学习目标数据集的判别边界。

#### 2. 目标数据集很小且与源数据集很相似时，应采用什么微调策略以防止过拟合？

可以采用保守的微调策略：冻结大部分甚至全部底层特征提取层，只训练新加入的顶层分类器；或者仅解冻靠近输出端的少数高层，并给这些预训练层设置较小学习率。与此同时，可以配合数据增强、权重衰减、早停和较少训练轮数。由于目标数据集与源数据集相似，预训练特征已经足够有效，过多更新全网络反而容易在小数据集上过拟合。


### 5.2 编程题：创建图像增广 Pipeline


In [4]:
if TORCH_AVAILABLE:
    augmentation_pipeline = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
        transforms.ToTensor(),
    ])

    print(augmentation_pipeline)
else:
    augmentation_pipeline = [
        "RandomResizedCrop(224, scale=(0.08, 1.0))",
        "RandomHorizontalFlip(p=0.5)",
        "ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)",
        "ToTensor()",
    ]
    print("PyTorch/torchvision 不可用；目标 transforms 管道为：")
    for step in augmentation_pipeline:
        print(step)


Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


## 6 目标检测，计算机视觉训练技巧

### 6.1 理论计算题

真实框 $A=[10,10,50,50]$，预测框 $B=[30,30,70,70]$，格式均为 $[x_1,y_1,x_2,y_2]$。

交集框左上角为

$$
(\max(10,30),\max(10,30))=(30,30)
$$

交集框右下角为

$$
(\min(50,70),\min(50,70))=(50,50)
$$

交集宽高分别为

$$
w=50-30=20,\quad h=50-30=20
$$

交集面积为

$$
S_{\cap}=20\times20=400
$$

两个框面积分别为

$$
S_A=(50-10)(50-10)=1600
$$

$$
S_B=(70-30)(70-30)=1600
$$

并集面积为

$$
S_{\cup}=S_A+S_B-S_{\cap}=1600+1600-400=2800
$$

因此 IoU 为

$$
IoU=\frac{S_{\cap}}{S_{\cup}}=\frac{400}{2800}=\frac{1}{7}\approx0.142857
$$

最终答案为

$$
\boxed{\frac{1}{7}\approx0.142857}
$$


### 6.2 编程题：实现标签平滑交叉熵损失


In [5]:
def label_smoothing_cross_entropy(logits, targets, epsilon=0.1, reduction="mean"):
    """计算标签平滑后的多分类交叉熵。targets 为类别下标。"""
    if TORCH_AVAILABLE:
        import torch.nn.functional as F

        if logits.dim() != 2:
            raise ValueError("logits must have shape (batch_size, num_classes)")
        num_classes = logits.size(1)
        if num_classes < 2:
            raise ValueError("num_classes must be at least 2")

        with torch.no_grad():
            smooth_labels = torch.full_like(logits, epsilon / (num_classes - 1))
            smooth_labels.scatter_(1, targets.unsqueeze(1), 1.0 - epsilon)

        log_probs = F.log_softmax(logits, dim=1)
        loss = -(smooth_labels * log_probs).sum(dim=1)
    else:
        logits = np.asarray(logits, dtype=float)
        targets = np.asarray(targets, dtype=int)
        if logits.ndim != 2:
            raise ValueError("logits must have shape (batch_size, num_classes)")
        num_classes = logits.shape[1]
        smooth_labels = np.full_like(logits, epsilon / (num_classes - 1), dtype=float)
        smooth_labels[np.arange(targets.shape[0]), targets] = 1.0 - epsilon
        shifted = logits - logits.max(axis=1, keepdims=True)
        log_probs = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
        loss = -(smooth_labels * log_probs).sum(axis=1)

    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    if reduction == "none":
        return loss
    raise ValueError("reduction must be 'mean', 'sum' or 'none'")


if TORCH_AVAILABLE:
    logits = torch.tensor([[3.0, 0.5, -1.0], [0.2, 1.5, 0.1]])
    targets = torch.tensor([0, 2])
else:
    logits = np.array([[3.0, 0.5, -1.0], [0.2, 1.5, 0.1]])
    targets = np.array([0, 2])
loss = label_smoothing_cross_entropy(logits, targets, epsilon=0.1)
print("loss:", float(loss))
print("per sample:", label_smoothing_cross_entropy(logits, targets, epsilon=0.1, reduction="none"))


loss: 1.0819056034088135
per sample: tensor([0.4207, 1.7431])
